A Gentle Introduction to torch.autograd

In [ ]:
import torch

# Create three scalars with requires_grad=True
# PyTorch will track operations on them to compute gradients later
a = torch.tensor(2.0, requires_grad=True)
b = torch.tensor(3.0, requires_grad=True)
x = torch.tensor(4.0, requires_grad=True)

# Forward pass: simple arithmetic
# y = a + b
# z = x * y
y = a + b
z = x * y

# Show results
print(f"y: {y}, z: {z}")

# grad_fn shows how this tensor was created
# It points to the operation that produced it
print(f"grad_fn for z: {z.grad_fn}")   # shows MulBackward0 because z came from multiplication
print(f"grad_fn for y: {y.grad_fn}")   # shows AddBackward0 because y came from addition
print(f"grad_fn for a: {a.grad_fn}")   # None because a is a leaf (input) tensor

# ------------------------------------------------------
# Element-wise multiplication (not matrix multiplication)
# * multiplies each element with the matching element
a = torch.tensor([[1, 2], [3, 4]])
b = torch.tensor([[10, 20], [30, 40]])

element_wise_product = a * b
print(f"Element-wise product:\n{element_wise_product}")
# Result:
# [[ 1*10, 2*20],
#  [3*30, 4*40]] = [[10, 40], [90, 160]]

# ------------------------------------------------------
# Matrix multiplication (dot product style)
# Use @ operator or torch.matmul()
w = torch.tensor([[1, 2, 4], [3, 4, 3]])
v = torch.tensor([[1, 2], [3, 4], [5, 6]])

matrix_product = w @ v
print(f"Matrix product:\n{matrix_product}")
# Shape: (2×3) @ (3×2) → (2×2)
# Each row of w is dotted with each column of v

# ------------------------------------------------------
# Reduction operations (turn many numbers into fewer)
scores = torch.tensor([[1.0, 20.0, 3.0], [4.0, 50.0, 6.0]])

# mean along dim=0 → average down each column (per assignment)
avg_per_assignment = scores.mean(dim=0)
print(f"Average per assignment:\n{avg_per_assignment}")
# Result: average of first column, second, third

# mean along dim=1 → average across each row (per student)
avg_per_student = scores.mean(dim=1)
print(f"Average per student:\n{avg_per_student}")
# Result: average of student 1, average of student 2

# ------------------------------------------------------
# argmax: find index of highest value along a dimension
best_indices = torch.argmax(scores, dim=1)
print(best_indices)
# Result: [1, 1] because 20 is max in row 0, 50 is max in row 1

# ------------------------------------------------------
# gather: pick values from tensor using given indices
data = torch.tensor([[10, 20, 30],
                     [40, 50, 60],
                     [70, 80, 90]])

# indices shape must match the output you want
# Here we want one value per row → indices has shape (3,1)
indices = torch.tensor([[2], [0], [1]])

# dim=1 means pick along columns for each row
gathered = torch.gather(data, dim=1, index=indices)
print(gathered)
# Result:
# row 0, column 2 → 30
# row 1, column 0 → 40
# row 2, column 1 → 80
# → tensor([[30], [40], [80]])

y: 5.0, z: 20.0
grad_fn for z: <MulBackward0 object at 0x000001A9BFFEA980>
grad_fn for y: <AddBackward0 object at 0x000001A9BFFE8040>
grad_fn for a: None
Element-wise product:
tensor([[ 10,  40],
        [ 90, 160]])
Matrix product:
tensor([[27, 34],
        [30, 40]])
Average per assignment:
tensor([ 2.5000, 35.0000,  4.5000])
Average per student:
tensor([ 8., 20.])
tensor([1, 1])
tensor([[30],
        [40],
        [80]])


In [ ]:
# Import PyTorch library (needed for tensors and automatic gradients)
import torch

# Set problem size
# N = number of data points (samples)
# D_in = number of input features (here 1 = simple x value)
# D_out = number of output features (here 1 = predict one y value)
N = 10
D_in = 1
D_out = 1

# Create random input data X (10 rows, 1 column each)
# These are your "study hours" or "input feature" values
X = torch.randn(N, D_in)

# These are the TRUE hidden parameters we want the model to learn
# true_w = correct slope (2.0)
# true_b = correct intercept / bias (1.0)
true_w = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)

# Create the real target values (what we want to predict)
# Formula: y = 2 × x + 1 + small random noise
# Noise makes data realistic (real world data is never perfect)
y_true = X @ true_w + true_b + torch.randn(N, D_out) * 0.1

# Create model parameters (our guesses)
# w = guessed slope, starts random
# b = guessed intercept, starts random
# requires_grad=True → PyTorch will track changes and compute gradients
w = torch.randn(D_in, D_out, requires_grad=True)
b = torch.randn(D_out, requires_grad=True)

# Forward pass: make prediction with current guesses
# y_pred = X multiplied by w + b
# This is your model's current best guess for every data point
y_pred = X @ w + b

# Show first 3 predictions and true values so you can compare
print(f"Predicted y: {y_pred[:3]}")
print(f"True y: {y_true[:3]}")

# Calculate how wrong each prediction is
error = y_pred - y_true

# Square the errors (makes bigger mistakes hurt more)
square_E = error ** 2

# Take average of all squared errors → this is your loss
# Lower loss = better model
loss = square_E.mean()
print(f"Loss: {loss}")

# Very important step: tell PyTorch to compute gradients
# This calculates how much changing w and b would reduce the loss
loss.backward()

# Now gradients are ready
# w.grad = how much to change w (negative = increase w to lower loss)
# b.grad = how much to change b
print(f"Gradient for w: {w.grad}")
print(f"Gradient for b: {b.grad}")

# Loss value is still the same (backward does not change loss)
print(f"Loss2: {loss}")

Predicted y: tensor([[2.0063],
        [1.3030],
        [1.2746]], grad_fn=<SliceBackward0>)
True y: tensor([[3.5066],
        [1.6524],
        [1.6358]])
Loss: 0.7404091358184814
Gradient for w: tensor([[-1.2144]])
Gradient for b: tensor([0.3984])
Loss2: 0.7404091358184814


In [3]:
import torch

# Example setup (same as your earlier code)
N = 10
X = torch.randn(N, 1)
true_w = torch.tensor([[2.0]])
true_b = torch.tensor(1.0)
y_true = X @ true_w + true_b + torch.randn(N, 1) * 0.1

# Your learnable parameters
w = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

# Hyperparameter you choose
learning_rate = 0.05

# One full training step (forward + backward + update)
# 1. Forward pass: make prediction
y_pred = X @ w + b

# 2. Compute loss (mean squared error)
loss = ((y_pred - y_true) ** 2).mean()

# 3. Backward pass: compute gradients
loss.backward()

# Show current values before update
print("Before update")
print("w:", w.item())
print("b:", b.item())
print("w.grad:", w.grad.item())
print("b.grad:", b.grad.item())
print("loss:", loss.item())

# 4. Update rule (this is the core gradient descent step)
# Subtract learning_rate * gradient from current parameter
# Use torch.no_grad() so PyTorch does NOT track these changes in the graph
with torch.no_grad():
    w_new = w - learning_rate * w.grad
    b_new = b - learning_rate * b.grad
    
    # Assign the new values back (you can also write w -= ... directly)
    w.copy_(w_new)
    b.copy_(b_new)

# 5. Reset gradients to zero for the next iteration
# If you skip this, gradients will accumulate (wrong behavior)
w.grad.zero_()
b.grad.zero_()

# Show values after update
print("\nAfter one update step")
print("w:", w.item())
print("b:", b.item())
print("loss would now be lower if you recompute it")

Before update
w: 0.3282260596752167
b: 0.756963312625885
w.grad: -3.1400997638702393
b.grad: 0.6911642551422119
loss: 2.5149903297424316

After one update step
w: 0.48523104190826416
b: 0.7224050760269165
loss would now be lower if you recompute it


In [ ]:
import torch
learning_rate,epochs = 0.1,100
w,b = torch.randn(1,1,requires_grad=True),torch.randn(1,1,requires_grad=True)
#train loop
for epoch in range(epochs):
    # use the input tensor defined earlier (capital X)
    y_hat = X @ w + b
    loss = ((y_hat - y_true) ** 2).mean()
    loss.backward()

    with torch.no_grad():
        w-=learning_rate*w.grad ;b-=learning_rate*b.grad
        w.grad.zero_(); b.grad.zero_()
        print(f"w: {w.item()}, b: {b.item()}, loss: {loss.item()}")
        
print(f"Final parameters: w={w.item()}, b={b.item()}")
print(f"True parameter: {loss.item()}")



NameError: name 'x' is not defined